In [20]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from google.colab import files
import re

In [12]:
#Header de la petición. X-Api-Key se toma de cualquier paquete de red con la página (Inspecionar elemento y network)
header ={
    'Content-type': 'application/json',
    'X-Api-Key': 'P1MfFHfQMOtL16Zpg36NcntJYCLFm8FqFfudnavl',
}
total= 357

#El request que se le hace al rest de metro cuadrado, por la url se pasan los parámetros. En este caso se buscan los apartmentos en PASADENA
#Al hacer el request directamente al back el archivo de respuesta es un json
r =requests.get("https://www.metrocuadrado.com/rest-search/search?realEstateTypeList=apartamento&realEstateBusinessList=venta&city=bogot%C3%A1&neighborhood=pasadena&from=50&size=50",headers=header)

In [13]:
r.json()

{'totalHits': 213,
 'totalEntries': 213,
 'geoSearch': False,
 'geoReferenced': None,
 'results': [{'contactPhone': '3118920084',
   'whatsapp': '573118920084',
   'title': 'Apartamento en Venta, PASADENA, Bogotá D.C.',
   'link': '/proyecto/well/21413-C0002-04',
   'imageLink': 'https://multimedia.metrocuadrado.com/21413-C0002-04/21413-C0002-04_1_V1_p.jpg',
   'whatsappMessage': 'Hola, estoy interesado en el anuncio con ID: 21413-C0002-04 que encontré en Metrocuadrado. Apartamento en Venta, PASADENA, Bogotá D.C. https://www.metrocuadrado.com',
   'badge': 'Proyecto de vivienda',
   'areaPrivada': 33.3,
   'midinmueble': '21413-C0002-04',
   'mtipoinmueble': {'id': '1', 'nombre': 'Apartamento'},
   'mtiponegocio': 'Venta',
   'mvalorventa': 490860000,
   'mvalorarriendo': None,
   'marea': 36.76,
   'mareac': 36.76,
   'areaprivada': 33.3,
   'mnrocuartos': '1',
   'mnrobanos': '2',
   'mnrogarajes': '0',
   'mciudad': {'id': '1', 'nombre': 'Bogotá D.C.'},
   'ptags': None,
   'mzona':

In [14]:
#Mismo proceso de arriba.

#En la petición anterior se ven total hits de 357, el total de apartamentos en venta en pasadena
total = 213
aptos = []

#Unicamente se pueden pedir un máximo de 50
for j in range(0, int(total/5)):
  url = str("https://www.metrocuadrado.com/rest-search/search?realEstateTypeList=apartamento&realEstateBusinessList=venta&city=bogot%C3%A1&neighborhood=pasadena&from="+str(int(j*50))+"&size=50")
  r =requests.get(url,headers=header)

  for i in r.json()['results']:
    #Se sacan algunos indicadores relevantes para análisis, valor total, metraje, barrio, link del detalle
    aptos.append({'ID':i['midinmueble'],'Link': i['link'],'Valor':i['mvalorventa'],'Area1':i['marea'],'Area2':i['mareac'],'Barrio': i['mbarrio'],   'Barrio2': i['mnombrecomunbarrio']})

aptos


[{'ID': '21413-C0002-02',
  'Link': '/proyecto/well/21413-C0002-02',
  'Valor': 348540000,
  'Area1': 25.53,
  'Area2': 25.53,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA'},
 {'ID': '21413-C0002-06',
  'Link': '/proyecto/well/21413-C0002-06',
  'Valor': 520000000,
  'Area1': 37.0,
  'Area2': 37.0,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA'},
 {'ID': '21413-C0002-03',
  'Link': '/proyecto/well/21413-C0002-03',
  'Valor': 448000000,
  'Area1': 37.0,
  'Area2': 37.0,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA'},
 {'ID': '21413-C0002-05',
  'Link': '/proyecto/well/21413-C0002-05',
  'Valor': 492360000,
  'Area1': 36.76,
  'Area2': 36.76,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA'},
 {'ID': '2671-M6258881',
  'Link': '/inmueble/venta-apartamento-bogota-pasadena-3-habitaciones-4-banos-2-garajes/2671-M6258881',
  'Valor': 690000000,
  'Area1': 114.33,
  'Area2': 106.0,
  'Barrio': 'PASADENA',
  'Barrio2': 'Pasadena'},
 {'ID': '20853-M6351825',
  'Link': '/inmueble/venta-apart

In [15]:
len(aptos)

207

In [21]:
def extraer_antiguedad(html: str):
    soup = BeautifulSoup(html, "html.parser")

    # 1) Intento 1: sitios Next antiguos (si existiera __NEXT_DATA__)
    s = soup.find("script", id="__NEXT_DATA__")
    if s and s.string:
        try:
            data = json.loads(s.string)
            return data["props"]["initialState"]["realestate"]["basic"].get("builtTime")
        except Exception:
            pass  # si cambia la ruta o no existe, caemos al fallback

    # 2) Fallback: Next App Router (datos en self.__next_f, escapados)
    m = re.search(r'\\"builtTime\\":\\"(.*?)\\"', html)
    return m.group(1) if m else None


for k in aptos:
    url2 = "https://www.metrocuadrado.com" + str(k["Link"])
    r = requests.get(url2, headers=header, timeout=20)
    r.raise_for_status()

    k["Antiguedad"] = extraer_antiguedad(r.text)


In [22]:
len(aptos)

207

In [25]:
aptos

[{'ID': '21413-C0002-02',
  'Link': '/proyecto/well/21413-C0002-02',
  'Valor': 348540000,
  'Area1': 25.53,
  'Area2': 25.53,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA',
  'Antiguedad': 'Para Estrenar'},
 {'ID': '21413-C0002-06',
  'Link': '/proyecto/well/21413-C0002-06',
  'Valor': 520000000,
  'Area1': 37.0,
  'Area2': 37.0,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA',
  'Antiguedad': 'Para Estrenar'},
 {'ID': '21413-C0002-03',
  'Link': '/proyecto/well/21413-C0002-03',
  'Valor': 448000000,
  'Area1': 37.0,
  'Area2': 37.0,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA',
  'Antiguedad': 'Para Estrenar'},
 {'ID': '21413-C0002-05',
  'Link': '/proyecto/well/21413-C0002-05',
  'Valor': 492360000,
  'Area1': 36.76,
  'Area2': 36.76,
  'Barrio': 'PASADENA',
  'Barrio2': 'PASADENA',
  'Antiguedad': 'Para Estrenar'},
 {'ID': '2671-M6258881',
  'Link': '/inmueble/venta-apartamento-bogota-pasadena-3-habitaciones-4-banos-2-garajes/2671-M6258881',
  'Valor': 690000000,
  'Area1': 11

In [23]:
#Descarga la información en un csv
df = pd.DataFrame(aptos)
df.to_csv('datos_completos.csv', index=False)
files.download("datos_completos.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>